# Bird Detection - FROM SCRATCH (yolov8n blueprint, no pretrained weights)

**This is NOT fine-tuning.** The fine-tuned V8 model started from
`yolov8n.pt` (weights pretrained on COCO - it inherited knowledge of edges,
feathers, shapes from millions of images; log said `Transferred 322/355
items from pretrained weights`). Here we build the same YOLOv8-nano
*architecture* from `yolov8n.yaml` - a text blueprint with **zero learned
knowledge** - and every weight starts as a random number. The model must
learn everything from our 950 bird images alone.

## How to run
1. Runtime -> Change runtime type -> **T4 GPU**
2. Runtime -> **Run all**
3. Wait ~1-1.5 h (250 epochs; it may stop earlier via early-stopping)
4. Results auto-save to Drive: `MyDrive/BIRD/scratch_v1/`

Reference numbers to beat (fine-tuned V8, same val set):
| model | epochs | mAP50 | mAP50-95 |
|---|---|---|---|
| V8 fine-tuned (COCO init) | 50 | 0.842 | 0.671 |
| V8 fine-tuned exported @448 (board production) | 50 | 0.806 | 0.587 |
| **this notebook: from scratch** | 250 | ? | ? |

> Expectation: from-scratch will score LOWER than fine-tuned on the same
> budget - that gap is exactly what we are measuring. It is the baseline
> for our own custom architecture.

In [ ]:
# Cell 1 - Mount Drive, find the dataset, copy to local SSD, fix labels
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

LOCAL_DATA = '/content/data_detection'

def find_dataset():
    candidates = [
        '/content/drive/MyDrive/BIRD/data_detection',
        '/content/drive/MyDrive/BIRD/bird_data_detection',
    ]
    for c in candidates:
        if os.path.isdir(os.path.join(c, 'train', 'images')):
            return c
    # deep fallback: any 'data_detection' folder under MyDrive (3 levels)
    hits = glob.glob('/content/drive/MyDrive/*/*/*/', recursive=False)
    for h in hits:
        if os.path.basename(h.rstrip('/')) == 'data_detection' and \
           os.path.isdir(os.path.join(h, 'train', 'images')):
            return h
    return None

src = find_dataset()
assert src, (
    'Dataset not found in Drive! Upload the folder\n'
    'training/V8 fine tuning/data_detection/  ->  Drive: MyDrive/BIRD/data_detection\n'
    '(the V8 notebook used the same location, so it is probably already there)'
)
print('Found dataset in Drive:', src)

if not os.path.exists(LOCAL_DATA):
    print('Copying to local SSD (fast)...')
    shutil.copytree(src, LOCAL_DATA)

# fix class ids: original Kaggle data used 24, we need 0 (idempotent)
fixed = 0
for lbl in glob.glob(os.path.join(LOCAL_DATA, '**', '*.txt'), recursive=True):
    with open(lbl) as f:
        lines = f.readlines()
    new_lines, changed = [], False
    for line in lines:
        parts = line.split()
        if parts and parts[0] != '0':
            parts[0] = '0'
            changed = True
        new_lines.append(' '.join(parts) + '\n')
    if changed:
        with open(lbl, 'w') as f:
            f.writelines(new_lines)
        fixed += 1

tr = len(os.listdir(os.path.join(LOCAL_DATA, 'train/images')))
va = len(os.listdir(os.path.join(LOCAL_DATA, 'val/images')))
print(f'Train: {tr}  Val: {va}  Labels fixed this run: {fixed}')

In [ ]:
# Cell 2 - install ultralytics
!pip install -q ultralytics

In [ ]:
# Cell 3 - dataset yaml (1 class: bird)
import yaml

data_yaml = {
    'path': '/content/data_detection',
    'train': 'train/images',
    'val': 'val/images',
    'names': {0: 'bird'},
}
with open('/content/bird_data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)
print(open('/content/bird_data.yaml').read())

# TRAIN FROM SCRATCH

`YOLO('yolov8n.yaml')` = **architecture only** (a blueprint describing
layers). No `.pt` file is downloaded, no weights are transferred. Combined
with `pretrained=False`, every parameter starts random.

Why 250 epochs instead of the fine-tune's 50? From scratch the model must
learn *low-level features* (edges, textures) that the COCO checkpoint
already knew, so it needs many more passes over our small dataset.

`save_period=50` keeps mid-run checkpoints as insurance against Colab
disconnects; `patience=30` stops early if val mAP stops improving.

In [ ]:
# Cell 4 - THE from-scratch training run
from ultralytics import YOLO

EPOCHS   = 250
IMGSZ    = 640
BATCH    = 16
PATIENCE = 30

# blueprint only -> random weights (NOT loading yolov8n.pt anywhere)
model = YOLO('yolov8n.yaml')
print('=== model built from blueprint; parameter count: ===')
model.info()

results = model.train(
    data='/content/bird_data.yaml',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    pretrained=False,          # <- the whole point: no pretrained weights
    project='/content/scratch_runs',
    name='yolov8n_scratch_v1',
    exist_ok=True,
    seed=0,
    save_period=50,            # checkpoint every 50 epochs (crash insurance)
)
print('Training done. Run dir:', model.trainer.save_dir)

In [ ]:
# Cell 5 - validate the from-scratch model on the val split
import glob as _g
from ultralytics import YOLO

best = _g.glob('/content/scratch_runs/*/weights/best.pt')[-1]
print('Validating:', best)
model = YOLO(best)
metrics = model.val(data='/content/bird_data.yaml', split='val')

print()
print('================ FROM-SCRATCH RESULT ================')
print(f"precision        : {metrics.box.mp:.3f}")
print(f"recall           : {metrics.box.mr:.3f}")
print(f"mAP50            : {metrics.box.map50:.3f}")
print(f"mAP50-95         : {metrics.box.map:.3f}")
print('-----------------------------------------------------')
print('fine-tuned V8 reference (same val): mAP50 0.842 / mAP50-95 0.671')
print('board production @448 export     : mAP50 0.806 / mAP50-95 0.587')

In [ ]:
# Cell 6 - learning curves + sample predictions
import glob, pandas as pd, matplotlib.pyplot as plt, cv2, os

run_dir = glob.glob('/content/scratch_runs/*')[-1]

# curves
df = pd.read_csv(os.path.join(run_dir, 'results.csv'))
df.columns = [c.strip() for c in df.columns]
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].plot(df['epoch'], df['train/box_loss'], label='box_loss')
ax[0].plot(df['epoch'], df['val/box_loss'], label='val_box_loss')
ax[0].set_title('box loss'); ax[0].legend()
ax[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50')
ax[1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95')
ax[1].set_title('accuracy'); ax[1].legend()
ax[2].plot(df['epoch'], df['metrics/precision(B)'], label='precision')
ax[2].plot(df['epoch'], df['metrics/recall(B)'], label='recall')
ax[2].set_title('P / R'); ax[2].legend()
ax[3].plot(df['epoch'], df['train/cls_loss'], label='cls_loss')
ax[3].plot(df['epoch'], df['train/dfl_loss'], label='dfl_loss')
ax[3].set_title('cls / dfl loss'); ax[3].legend()
plt.tight_layout(); plt.savefig('/content/scratch_curves.png', dpi=110); plt.show()

# predictions on a few val images
val_imgs = sorted(glob.glob('/content/data_detection/val/images/*.jpg'))[:6]
model.predict(val_imgs, conf=0.25, save=True, project='/content/preds', name='scratch_v1', exist_ok=True)
fig, ax = plt.subplots(2, 3, figsize=(16, 10))
for i, p in enumerate(sorted(glob.glob('/content/preds/scratch_v1/*.jpg'))[:6]):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax[i//3][i%3].imshow(img); ax[i//3][i%3].axis('off')
plt.tight_layout(); plt.savefig('/content/scratch_preds.png', dpi=110); plt.show()

In [ ]:
# Cell 7 - persist everything to Drive (survives the runtime ending)
import glob, shutil, os

DST = '/content/drive/MyDrive/BIRD/scratch_v1'
os.makedirs(DST, exist_ok=True)
run_dir = glob.glob('/content/scratch_runs/*')[-1]

keep = [
    'weights/best.pt', 'weights/last.pt', 'results.csv', 'args.yaml',
    'results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
    'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png',
    'labels.jpg', 'labels_correlogram.jpg',
]
for rel in keep:
    src = os.path.join(run_dir, rel)
    if os.path.exists(src):
        shutil.copy2(src, DST)

for f in ('/content/scratch_curves.png', '/content/scratch_preds.png'):
    if os.path.exists(f):
        shutil.copy2(f, DST)

# also snapshot epoch checkpoints (epoch50/100/... .pt) if any exist
for ck in glob.glob(os.path.join(run_dir, 'weights', 'epoch*.pt')):
    shutil.copy2(ck, DST)

print('Saved to', DST)
for f in sorted(os.listdir(DST)):
    print(' ', f, f'{os.path.getsize(os.path.join(DST,f))/1e6:.1f} MB')
print()
print('DONE. The PC session picks this up from MyDrive/BIRD/scratch_v1/')

---
## OPTIONAL - GPU bridge for the PC session (skip during a normal run)

Run the cell below **only if the PC OpenCode session asks for interactive
GPU access** (custom-model iteration phase). It starts an SSH server in
this Colab VM and publishes it through a temporary Cloudflare tunnel, then
prints one URL - paste that URL back into the PC session. Nothing else in
this notebook depends on it.

In [ ]:
# Cell 8 - OPTIONAL ssh tunnel (run only when the PC session requests it)
import os, re, time, subprocess

print('Installing sshd...')
subprocess.run('apt-get install -y -q openssh-server > /dev/null 2>&1', shell=True)
subprocess.run('mkdir -p -m700 /var/run/sshd', shell=True)
subprocess.run('echo root:birdgpu | chpasswd', shell=True)
subprocess.run('service ssh start', shell=True)

if not os.path.exists('/content/cf'):
    subprocess.run(
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
        'cloudflared-linux-amd64 -O /content/cf && chmod +x /content/cf', shell=True)

subprocess.run('pkill -f cf tunnel', shell=True)
subprocess.Popen(
    '/content/cf tunnel --url ssh://localhost:22 > /content/cf.log 2>&1',
    shell=True)
time.sleep(8)
log = open('/content/cf.log').read()
m = re.search(r'https://\S*trycloudflare\.com', log)
if m:
    print()
    print('=' * 60)
    print('PASTE THIS LINE TO THE PC SESSION:')
    print(m.group(0))
    print('(ssh user root, password birdgpu)')
    print('=' * 60)
else:
    print('tunnel not ready yet - open /content/cf.log and look for a')
    print('https://...trycloudflare.com URL, paste it to the PC session.')

: 